# Rerun `generate_graphs.py` with `mntss/clt-gemma-2-2b-426k`

This notebook reruns the repository script with the 426k Gemma 2 2B CLT transcoder. It works from the repo checkout or from a fresh `/content` runtime by cloning the repo first.

In [3]:
!nvidia-smi

Thu Jun 25 16:10:33 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX PRO 6000 Blac...    Off |   00000000:05:00.0 Off |                    0 |
| N/A   31C    P0             45W /  600W |       0MiB /  97887MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [4]:
from pathlib import Path
import subprocess
import sys

REPO_URL = "https://github.com/IamKrill1n/circuit_tracer_mod.git"
REPO_NAME = "circuit_tracer_mod"
REPO_BRANCH = "clean_up"

def find_repo_root() -> Path | None:
    candidates = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    candidates.extend([Path("/home/tu/circuit_tracer_mod"), Path("/content") / REPO_NAME])
    for candidate in candidates:
        if (candidate / "generate_graphs.py").exists():
            return candidate
    return None

REPO_ROOT = find_repo_root()
clone_parent = Path("/content") if Path("/content").exists() else Path.cwd().resolve()
if REPO_ROOT is None:
    REPO_ROOT = clone_parent / REPO_NAME
    subprocess.run(
        ["git", "clone", "--branch", REPO_BRANCH, REPO_URL, str(REPO_ROOT)], check=True
    )
elif REPO_ROOT == clone_parent / REPO_NAME and (REPO_ROOT / ".git").exists():
    subprocess.run(["git", "fetch", "origin", REPO_BRANCH], cwd=REPO_ROOT, check=True)
    subprocess.run(["git", "checkout", REPO_BRANCH], cwd=REPO_ROOT, check=True)

SCRIPT = REPO_ROOT / "generate_graphs.py"
assert SCRIPT.exists(), f"missing {SCRIPT}"

print(f"repo: {REPO_ROOT}")
print(f"branch: {REPO_BRANCH}")

repo: /content/circuit_tracer_mod
branch: clean_up


If this is a fresh Colab runtime, run the next cell once to install the repo dependencies. Skip it when you are already in the local `circuit` conda environment.

In [5]:
INSTALL_DEPS = Path("/content").exists()

if INSTALL_DEPS:
    subprocess.run([sys.executable, "-m", "pip", "install", "-e", str(REPO_ROOT)], check=True)

In [6]:
PROMPT_FILE = REPO_ROOT / "dataset" / "analogies" / "bats_analogies.txt"
OUTPUT_DIR = REPO_ROOT / "dataset" / "analogies" / "graphs" / "clt-gemma-2-2b-426k"

MODEL = "google/gemma-2-2b"
TRANSCODER = "mntss/clt-gemma-2-2b-426k"
BACKEND = "transformerlens"
MAX_N_LOGITS = 15
DESIRED_LOGIT_PROB = 0.99
HF_REPO = None

assert PROMPT_FILE.exists(), f"missing {PROMPT_FILE}"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"prompt file: {PROMPT_FILE}")
print(f"output dir: {OUTPUT_DIR}")

AssertionError: missing /content/circuit_tracer_mod/dataset/analogies/bats_analogies.txt

In [ ]:
import shutil

if shutil.which("conda"):
    python_cmd = ["conda", "run", "-n", "circuit", "python"]
else:
    python_cmd = [sys.executable]

cmd = [
    *python_cmd,
    str(SCRIPT),
    "--prompt-file",
    str(PROMPT_FILE),
    "--output-dir",
    str(OUTPUT_DIR),
    "--model",
    MODEL,
    "--transcoder",
    TRANSCODER,
    "--backend",
    BACKEND,
    "--max-n-logits",
    str(MAX_N_LOGITS),
    "--desired-logit-prob",
    str(DESIRED_LOGIT_PROB),
]

if HF_REPO:
    cmd.extend(["--hf-repo", HF_REPO])

print(" ".join(cmd))

In [ ]:
subprocess.run(cmd, cwd=REPO_ROOT, check=True)